<a href="https://www.kaggle.com/code/jatinnabhoya/radiology-report-rag-01-data-prep?scriptVersionId=370145547" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# load and merge

In [1]:
import pandas as pd, numpy as np, re, os

DATA = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university"
IMG_DIR = f"{DATA}/images/images_normalized"
reports = pd.read_csv(f"{DATA}/indiana_reports.csv")
proj = pd.read_csv(f"{DATA}/indiana_projections.csv")
print(reports.columns.tolist())
print(proj["projection"].value_counts())

['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']
projection
Frontal    3818
Lateral    3648
Name: count, dtype: int64


# keep one frontal image per study with non-empty Findings


In [2]:
df = proj[proj["projection"] == "Frontal"].merge(reports, on="uid")
df = df.drop_duplicates("uid", keep="first")          # one frontal X-ray per study

def clean(t):
    t = re.sub(r"\bX+\b", "", str(t))                  # remove de-identification tokens (XXXX)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df = df[df["findings"].notna()].copy()
df["findings_clean"] = df["findings"].map(clean)
df = df[df["findings_clean"].str.split().str.len() >= 3]   # drop near-empty findings
df["image_path"] = IMG_DIR + "/" + df["filename"]
df["is_abnormal"] = (df["Problems"].str.strip().str.lower() != "normal").astype(int)

print("studies kept:", len(df))
print("abnormal share: %.1f%%" % (100 * df["is_abnormal"].mean()))
print("findings length (words):", df["findings_clean"].str.split().str.len().describe().round(1).to_dict())

studies kept: 3199
abnormal share: 63.6%
findings length (words): {'count': 3199.0, 'mean': 30.6, 'std': 14.0, 'min': 7.0, '25%': 21.0, '50%': 29.0, '75%': 37.0, 'max': 167.0}


# 70 / 10 / 20 split, with the same normal/abnormal share in each part

In [3]:
from sklearn.model_selection import train_test_split

train, rest = train_test_split(df, test_size=0.30, random_state=42, stratify=df["is_abnormal"])
val, test = train_test_split(rest, test_size=2/3, random_state=42, stratify=rest["is_abnormal"])
train, val, test = train.assign(split="train"), val.assign(split="val"), test.assign(split="test")
clean_df = pd.concat([train, val, test])[["uid", "filename", "image_path", "findings_clean", "is_abnormal", "split"]]
clean_df = clean_df.rename(columns={"findings_clean": "findings"})

print(clean_df["split"].value_counts())
print(clean_df.groupby("split")["is_abnormal"].mean().round(3))
assert clean_df["uid"].is_unique   # each study in exactly one split, so no leakage

clean_df.to_csv("/kaggle/working/iu_xray_clean.csv", index=False)
clean_df.head(3)

split
train    2239
test      640
val       320
Name: count, dtype: int64
split
test     0.636
train    0.636
val      0.638
Name: is_abnormal, dtype: float64


,uid,filename,image_path,findings,is_abnormal,split
1724,1796,1796_IM-0517-1002.dcm.png,/kaggle/input/datasets/raddar/chest-xrays-indi...,"The lungs are clear bilaterally. Specifically,...",0,train
238,248,248_IM-1008-85149002.dcm.png,/kaggle/input/datasets/raddar/chest-xrays-indi...,"No pneumothorax, pleural effusion or airspace ...",1,train
2101,2198,2198_IM-0808-1001.dcm.png,/kaggle/input/datasets/raddar/chest-xrays-indi...,"Cardiac silhouette, pulmonary vascular pattern...",0,train


# save to Hugging Face (private dataset repo)

In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
user = api.whoami()["name"]
REPO = f"{user}/Radiology_Report_RAG-data"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="/kaggle/working/iu_xray_clean.csv",
                path_in_repo="iu_xray_clean.csv", repo_id=REPO, repo_type="dataset")
print("Uploaded to https://huggingface.co/datasets/" + REPO)

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to https://huggingface.co/datasets/jatinnabhoya/Radiology_Report_RAG-data
